# DQN Reward Weight Search

Finds optimal reward weights using **random search** vs **Bayesian optimisation** (Optuna).  
Training: `data/beginner_train.npz` — Evaluation: `data/beginner_test.npz` (held-out).  
Action space: `0..80` = reveal, `81..161` = flag/unflag.  
`mine_penalty` is fixed at −10 as the anchor; all other weights are searched.

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "matplotlib", "imageio", "optuna", "tqdm"],
    check=True
)

sys.path.insert(0, "..")

import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import imageio
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from dataclasses import dataclass, asdict
from tqdm.notebook import tqdm
from IPython.display import Image as IPImage, display

from src.env.minesweeper_env import MinesweeperEnv
from src.train.dqn import DQNAgent

print(f"torch {torch.__version__}  |  optuna {optuna.__version__}")
print("All imports OK")

## Reward Config & Search Space

In [ ]:
@dataclass
class RewardConfig:
    mine_penalty:        float = -10.0  # fixed anchor — do not search
    reveal_reward:       float =   1.0  # per newly revealed safe cell
    win_bonus:           float =  50.0  # on winning
    step_reward:         float =   0.0  # per step survived (survival bonus)
    correct_flag_reward: float =   0.0  # immediate reward for flagging a mine
    wrong_flag_penalty:  float =   0.0  # immediate penalty for flagging a safe cell


# Search bounds for the 5 tunable weights
SEARCH_SPACE = {
    "reveal_reward":       (0.0,  2.0),
    "win_bonus":           (10.0, 100.0),
    "step_reward":         (0.0,  0.2),
    "correct_flag_reward": (0.0,  5.0),
    "wrong_flag_penalty":  (0.0,  5.0),
}

## Reward Wrapper

In [ ]:
class RewardWrapper:
    """Wraps MinesweeperEnv: reshapes rewards per RewardConfig and adds flag actions."""

    def __init__(self, env: MinesweeperEnv, cfg: RewardConfig):
        self._env        = env
        self.cfg         = cfg
        self.rows        = env.rows
        self.cols        = env.cols
        self.action_size = env.action_size * 2  # 0..N-1 reveal | N..2N-1 flag
        self.state_size  = env.state_size
        self._step       = 0

    def reset(self):
        self._step = 0
        return self._env.reset()

    def get_valid_actions(self):
        board   = self._env._game.board
        N       = self._env.action_size
        hidden  = board.hidden_cells()
        flagged = [i for i, v in enumerate(board.view.flatten()) if v == 9]
        return hidden + [i + N for i in hidden + flagged]

    def step(self, action):
        N     = self._env.action_size
        board = self._env._game.board
        self._step += 1

        if action >= N:
            # ---- Flag / unflag ----
            cell        = action - N
            row, col    = divmod(cell, self.cols)
            was_flagged = board.view[row, col] == 9
            self._env._game.flag(row, col)
            now_flagged = board.view[row, col] == 9

            reward = 0.0
            if board._mines_placed and now_flagged and not was_flagged:
                reward = (self.cfg.correct_flag_reward if board.is_mine(row, col)
                          else -self.cfg.wrong_flag_penalty)

            return (board.get_observation(), reward, self._env._game.is_over,
                    {"state": self._env._game.state.name, "result": "flag",
                     "mines_left": board.mines_remaining()})

        # ---- Reveal ----
        obs_before = board.get_observation()
        obs, _, done, info = self._env.step(action)
        newly = (int(((obs >= 0) & (obs <= 8)).sum()) -
                 int(((obs_before >= 0) & (obs_before <= 8)).sum()))

        reward = 0.0
        if info["result"] == "mine":
            reward = self.cfg.mine_penalty
        elif info["result"] != "already_revealed":
            reward = newly * self.cfg.reveal_reward + self.cfg.step_reward
            if done and info["state"] == "WON":
                reward += self.cfg.win_bonus

        return obs, reward, done, info

## Training / Evaluation Function

Three training completeness signals are tracked per trial:
- **`ReduceLROnPlateau`** — halves the learning rate when eval win rate stops improving (patience=3 checkpoints). When LR hits the floor the network has effectively converged.
- **Gradient norm** — mean L2 norm of all parameter gradients per episode. Near-zero norms mean the network has stopped learning regardless of LR.
- **Early stopping** — halts training if eval win rate hasn't improved in `ES_PATIENCE` consecutive checkpoints, saving time during the weight search.

In [ ]:
import torch.optim.lr_scheduler as lr_sched

TRAIN_DATASET = "../data/beginner_train.npz"
EVAL_DATASET  = "../data/beginner_test.npz"
ES_PATIENCE   = 3   # early stop after this many non-improving eval checkpoints

def run_trial(
    cfg,
    n_episodes    = 200,
    eval_every    = 50,    # evaluate every 50 eps
    eval_episodes = 100,
):
    """Train a fresh DQN with cfg. Returns final test win rate, eval history,
    and per-episode diagnostics (grad norms, LR, early-stop flag)."""

    env   = RewardWrapper(MinesweeperEnv(preset="beginner", dataset=TRAIN_DATASET), cfg)
    agent = DQNAgent(state_size=env.state_size, action_size=env.action_size)

    scheduler = lr_sched.ReduceLROnPlateau(
        agent.optimiser, mode="max", factor=0.5, patience=3, min_lr=1e-5
    )

    eval_win_rates = []
    grad_norm_log  = []
    lr_log         = []

    best_eval     = 0.0
    no_improve    = 0
    stopped_early = False

    pbar = tqdm(range(n_episodes), desc="training", unit="ep", leave=False)

    for ep in pbar:
        obs, done = env.reset(), False
        ep_grad_norms = []

        while not done:
            action = agent.choose_action(obs, env.get_valid_actions())
            next_obs, reward, done, info = env.step(action)
            agent.push(obs, action, reward, next_obs, done)
            result = agent.train_step()
            if result is not None:
                loss, grad_norm = result
                ep_grad_norms.append(grad_norm)
            obs = next_obs

        grad_norm_log.append(float(np.mean(ep_grad_norms)) if ep_grad_norms else float("nan"))

        pbar.set_postfix(
            wr=f"{best_eval:.3f}",
            lr=f"{agent.optimiser.param_groups[0]['lr']:.1e}",
        )

        if (ep + 1) % eval_every == 0:
            pbar.set_description("evaluating")
            saved = agent.epsilon
            agent.epsilon = 0.0
            eval_env = RewardWrapper(
                MinesweeperEnv(preset="beginner", dataset=EVAL_DATASET), cfg
            )
            wins = 0
            for _ in range(eval_episodes):
                o, d = eval_env.reset(), False
                while not d:
                    a = agent.choose_action(o, eval_env.get_valid_actions())
                    o, _, d, i = eval_env.step(a)
                wins += i["state"] == "WON"
            agent.epsilon = saved

            wr = wins / eval_episodes
            eval_win_rates.append(wr)
            current_lr = agent.optimiser.param_groups[0]["lr"]
            lr_log.append(current_lr)
            scheduler.step(wr)

            if wr > best_eval:
                best_eval  = wr
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= ES_PATIENCE:
                    stopped_early = True
                    pbar.set_description("early stop")
                    pbar.close()
                    break

            pbar.set_description("training")

    if not stopped_early:
        pbar.close()

    final_wr = eval_win_rates[-1] if eval_win_rates else 0.0
    return final_wr, eval_win_rates, agent, {
        "grad_norm_log": grad_norm_log,
        "lr_log":        lr_log,
        "stopped_early": stopped_early,
        "episodes_run":  (ep + 1),
    }

## Search Settings

Defaults are tuned for **< 5 min total** on CPU (both search methods combined).  
Scale up `N_TRIALS` / `N_EPISODES` once you're happy with the setup.

In [ ]:
N_TRIALS   = 10    # trials per search method  (scale up later)
N_EPISODES = 200   # episodes per trial        (scale up later)

## Random Search

In [ ]:
rng = np.random.default_rng(42)

random_trials      = []
random_best_so_far = []

for t in range(N_TRIALS):
    weights = {k: float(rng.uniform(lo, hi))
               for k, (lo, hi) in SEARCH_SPACE.items()}
    cfg = RewardConfig(**weights)

    final_wr, history, _, diag = run_trial(cfg, n_episodes=N_EPISODES)

    random_trials.append({"cfg": cfg, "final_wr": final_wr, "history": history, "diag": diag})
    best_so_far = max(r["final_wr"] for r in random_trials)
    random_best_so_far.append(best_so_far)
    flag = " [early stop]" if diag["stopped_early"] else ""
    print(f"  [random] trial {t+1:>3}/{N_TRIALS}  "
          f"wr={final_wr:.3f}  best={best_so_far:.3f}  "
          f"eps={diag['episodes_run']:>6}  "
          f"lr={diag['lr_log'][-1]:.2e}{flag}")

best_random = max(random_trials, key=lambda r: r["final_wr"])
print(f"\nRandom search best: {best_random['final_wr']:.3f}")

## Bayesian Search (Optuna TPE)

In [ ]:
bayes_trials      = []
bayes_best_so_far = []

def objective(trial):
    cfg = RewardConfig(
        reveal_reward       = trial.suggest_float("reveal_reward",       *SEARCH_SPACE["reveal_reward"]),
        win_bonus           = trial.suggest_float("win_bonus",           *SEARCH_SPACE["win_bonus"]),
        step_reward         = trial.suggest_float("step_reward",         *SEARCH_SPACE["step_reward"]),
        correct_flag_reward = trial.suggest_float("correct_flag_reward", *SEARCH_SPACE["correct_flag_reward"]),
        wrong_flag_penalty  = trial.suggest_float("wrong_flag_penalty",  *SEARCH_SPACE["wrong_flag_penalty"]),
    )
    final_wr, history, _, diag = run_trial(cfg, n_episodes=N_EPISODES)

    bayes_trials.append({"cfg": cfg, "final_wr": final_wr, "history": history, "diag": diag})
    best_so_far = max(r["final_wr"] for r in bayes_trials)
    bayes_best_so_far.append(best_so_far)
    t = len(bayes_trials)
    flag = " [early stop]" if diag["stopped_early"] else ""
    print(f"  [bayes]  trial {t:>3}/{N_TRIALS}  "
          f"wr={final_wr:.3f}  best={best_so_far:.3f}  "
          f"eps={diag['episodes_run']:>6}  "
          f"lr={diag['lr_log'][-1]:.2e}{flag}")
    return final_wr

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS)

best_bayes = max(bayes_trials, key=lambda r: r["final_wr"])
print(f"\nBayesian search best: {best_bayes['final_wr']:.3f}")

## Comparison Plots

In [ ]:
os.makedirs("../recordings", exist_ok=True)

all_random_wrs = [r["final_wr"] for r in random_trials]
all_bayes_wrs  = [r["final_wr"] for r in bayes_trials]
weight_keys    = list(SEARCH_SPACE.keys())

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Random Search vs Bayesian Optimisation — DQN Reward Weights", fontsize=14)

# ---- 1. Best-so-far convergence ----
ax = fig.add_subplot(2, 3, 1)
ax.plot(range(1, N_TRIALS + 1), [v * 100 for v in random_best_so_far],
        color="steelblue", lw=2, marker="o", ms=4, label="Random")
ax.plot(range(1, N_TRIALS + 1), [v * 100 for v in bayes_best_so_far],
        color="tomato",   lw=2, marker="s", ms=4, label="Bayesian (TPE)")
ax.set_title("Best-So-Far Test Win Rate")
ax.set_xlabel("Trial")
ax.set_ylabel("Win rate (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# ---- 2. All trial win rates (scatter) ----
ax = fig.add_subplot(2, 3, 2)
ax.scatter(range(1, N_TRIALS + 1), [v * 100 for v in all_random_wrs],
           color="steelblue", alpha=0.7, label="Random", zorder=3)
ax.scatter(range(1, N_TRIALS + 1), [v * 100 for v in all_bayes_wrs],
           color="tomato",   alpha=0.7, label="Bayesian", marker="s", zorder=3)
ax.set_title("All Trial Win Rates")
ax.set_xlabel("Trial")
ax.set_ylabel("Test win rate (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# ---- 3. Distribution comparison (box plot) ----
ax = fig.add_subplot(2, 3, 3)
bp = ax.boxplot(
    [[v * 100 for v in all_random_wrs], [v * 100 for v in all_bayes_wrs]],
    labels=["Random", "Bayesian"],
    patch_artist=True,
    medianprops={"color": "black", "lw": 2},
)
bp["boxes"][0].set_facecolor("steelblue")
bp["boxes"][1].set_facecolor("tomato")
for patch in bp["boxes"]:
    patch.set_alpha(0.6)
ax.set_title("Win Rate Distribution")
ax.set_ylabel("Test win rate (%)")
ax.grid(True, alpha=0.3, axis="y")

# ---- 4–6. Weight vs win rate scatter (top 3 most important weights) ----
for idx, key in enumerate(weight_keys[:3]):
    ax = fig.add_subplot(2, 3, 4 + idx)
    rand_x = [r["cfg"].__dict__[key] for r in random_trials]
    bayes_x = [r["cfg"].__dict__[key] for r in bayes_trials]
    ax.scatter(rand_x,  [v * 100 for v in all_random_wrs],
               color="steelblue", alpha=0.7, label="Random", zorder=3)
    ax.scatter(bayes_x, [v * 100 for v in all_bayes_wrs],
               color="tomato",   alpha=0.7, label="Bayesian", marker="s", zorder=3)
    lo, hi = SEARCH_SPACE[key]
    ax.set_xlim(lo, hi)
    ax.set_title(f"{key} vs win rate")
    ax.set_xlabel(key)
    ax.set_ylabel("Test win rate (%)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../recordings/search_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/search_comparison.png")

## Optuna Parameter Importance

In [ ]:
# Use best overall config's diagnostics
best_overall = best_bayes if best_bayes["final_wr"] >= best_random["final_wr"] else best_random
diag = best_overall["diag"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Training Completeness — Best Config", fontsize=13)

# ---- 1. Gradient norm over training ----
ax = axes[0]
gnorms = [g for g in diag["grad_norm_log"] if not np.isnan(g)]
window = min(100, len(gnorms) // 5 or 1)
smoothed = np.convolve(gnorms, np.ones(window) / window, mode="valid")
ax.plot(smoothed, color="steelblue", lw=1)
ax.set_title("Gradient Norm (smoothed)")
ax.set_xlabel("Episode")
ax.set_ylabel("L2 norm")
ax.grid(True, alpha=0.3)
# Near-zero = network stopped learning
ax.axhline(0.01, color="tomato", lw=1, ls="--", label="Near-zero threshold")
ax.legend(fontsize=8)

# ---- 2. Learning rate schedule ----
ax = axes[1]
eval_xs = [500 * (i + 1) for i in range(len(diag["lr_log"]))]
ax.plot(eval_xs, diag["lr_log"], color="darkorange", lw=2, marker="o", ms=4)
ax.set_title("Learning Rate (ReduceLROnPlateau)")
ax.set_xlabel("Episode")
ax.set_ylabel("LR")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
# Mark where LR reductions happened
lrs = diag["lr_log"]
for i in range(1, len(lrs)):
    if lrs[i] < lrs[i - 1]:
        ax.axvline(eval_xs[i], color="tomato", lw=1, ls="--", alpha=0.6)

# ---- 3. Early stopping across all trials ----
ax = axes[2]
rand_eps  = [r["diag"]["episodes_run"] for r in random_trials]
bayes_eps = [r["diag"]["episodes_run"] for r in bayes_trials]
rand_stopped  = sum(r["diag"]["stopped_early"] for r in random_trials)
bayes_stopped = sum(r["diag"]["stopped_early"] for r in bayes_trials)
ax.hist(rand_eps,  bins=10, alpha=0.6, color="steelblue",
        label=f"Random ({rand_stopped}/{N_TRIALS} early stopped)")
ax.hist(bayes_eps, bins=10, alpha=0.6, color="tomato",
        label=f"Bayesian ({bayes_stopped}/{N_TRIALS} early stopped)")
ax.axvline(N_EPISODES, color="black", lw=1.2, ls="--", label=f"Max ({N_EPISODES})")
ax.set_title("Episodes Run per Trial")
ax.set_xlabel("Episodes")
ax.set_ylabel("Count")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../recordings/training_completeness.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/training_completeness.png")
print(f"\nBest config ran {diag['episodes_run']} / {N_EPISODES} episodes  "
      f"({'early stopped' if diag['stopped_early'] else 'ran to completion'})")
print(f"Final LR: {diag['lr_log'][-1]:.2e}  |  "
      f"Final mean grad norm: {np.nanmean(diag['grad_norm_log'][-500:]):.4f}")

## Training Completeness Diagnostics (Best Config)

In [ ]:
try:
    importances = optuna.importance.get_param_importances(study)
    fig, ax = plt.subplots(figsize=(8, 4))
    keys = list(importances.keys())
    vals = [importances[k] for k in keys]
    bars = ax.barh(keys, vals, color="tomato", edgecolor="white", alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(v + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{v:.3f}", va="center", fontsize=9)
    ax.set_title("Optuna Parameter Importance (Bayesian trials)")
    ax.set_xlabel("Importance score")
    ax.set_xlim(0, max(vals) * 1.2)
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig("../recordings/param_importance.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: recordings/param_importance.png")
except Exception as e:
    print(f"Importance plot unavailable: {e}")

## Best Configs Summary

In [ ]:
print("=" * 56)
print("  RANDOM SEARCH — best config")
print("=" * 56)
print(f"  Test win rate:       {best_random['final_wr']*100:.2f}%")
for k in weight_keys:
    print(f"  {k:<24} {best_random['cfg'].__dict__[k]:.4f}")

print()
print("=" * 56)
print("  BAYESIAN SEARCH — best config")
print("=" * 56)
print(f"  Test win rate:       {best_bayes['final_wr']*100:.2f}%")
for k in weight_keys:
    print(f"  {k:<24} {best_bayes['cfg'].__dict__[k]:.4f}")

print()
print("=" * 56)
print("  HEAD-TO-HEAD")
print("=" * 56)
print(f"  Random   — mean: {np.mean(all_random_wrs)*100:.2f}%  "
      f"max: {max(all_random_wrs)*100:.2f}%  "
      f"std: {np.std(all_random_wrs)*100:.2f}%")
print(f"  Bayesian — mean: {np.mean(all_bayes_wrs)*100:.2f}%  "
      f"max: {max(all_bayes_wrs)*100:.2f}%  "
      f"std: {np.std(all_bayes_wrs)*100:.2f}%")

## GIF — Best Overall Config

In [ ]:
import pygame
pygame.init()
from src.game.renderer import Renderer

# Pick whichever search found the higher win rate
if best_bayes["final_wr"] >= best_random["final_wr"]:
    winner_label, winner = "Bayesian", best_bayes
else:
    winner_label, winner = "Random", best_random
print(f"Recording episode using {winner_label} best config "
      f"(test wr={winner['final_wr']*100:.1f}%)")

# Re-train a fresh agent with the best config for recording
_, _, rec_agent = run_trial(winner["cfg"], n_episodes=N_EPISODES)
rec_agent.epsilon = 0.0  # greedy

rec_base = MinesweeperEnv(preset="beginner", dataset=EVAL_DATASET)
rec_env  = RewardWrapper(rec_base, winner["cfg"])
obs      = rec_env.reset()

rec_renderer = Renderer(rec_base._game, preset="beginner")

def capture():
    rec_renderer._draw()
    return pygame.surfarray.array3d(rec_renderer.screen).transpose(1, 0, 2).copy()

frames = [capture()]
done   = False
while not done:
    action = rec_agent.choose_action(obs, rec_env.get_valid_actions())
    obs, _, done, info = rec_env.step(action)
    frames.append(capture())

GIF_PATH = "../recordings/dqn_best_episode.gif"
imageio.mimsave(GIF_PATH, frames, duration=0.4, loop=0)
print(f"Outcome: {info['state']} in {len(frames)-1} steps")
print(f"Saved: {GIF_PATH}")
display(IPImage(filename=GIF_PATH))

## Interpretation

### What to look for

**Best-so-far convergence plot** — if Bayesian finds a high win rate in fewer trials than random, TPE is exploiting structure in the reward space. If the lines converge to the same value, the landscape is flat enough that random sampling is sufficient.

**Distribution box plot** — a higher median for Bayesian means it's consistently finding better configs, not just getting lucky once. If the distributions overlap heavily, consider running more trials.

**Weight vs win rate scatter** — a clear trend (e.g. higher `reveal_reward` → higher win rate) confirms that weight matters and Bayesian should be concentrating samples there. A flat scatter means the weight doesn't move the needle much.

**Parameter importance** — Optuna's importance score (fANOVA) measures how much variance in win rate each weight explains. Low-importance weights can be fixed at their best-found value in a follow-up fine search over just the high-importance ones.

### Follow-up

After running this notebook, fix the low-importance weights and run a second Bayesian search over just the top 2–3 weights with tighter bounds. This is the most efficient path to a well-tuned reward function before moving to a more complex architecture.